In [1]:
import requests

def mesh_english_aliases(mesh_id: str) -> dict:
    url = f"https://id.nlm.nih.gov/mesh/{mesh_id}.jsonld"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    data = r.json()
    graph = data.get("@graph", [])
    uri = f"https://id.nlm.nih.gov/mesh/{mesh_id}"
    main = next((n for n in graph if n.get("@id") == uri), None) or {}

    def labels(field):
        vals = main.get(field, [])
        if not isinstance(vals, list):
            vals = [vals]
        out = []
        for v in vals:
            if isinstance(v, dict) and v.get("@language") == "en":
                out.append(v.get("@value"))
        return [x for x in out if isinstance(x, str)]

    pref = labels("http://www.w3.org/2004/02/skos/core#prefLabel")
    alt = labels("http://www.w3.org/2004/02/skos/core#altLabel")

    preferred = pref[0] if pref else None
    entry_terms = []
    seen = set()
    for t in alt:
        if t != preferred and t not in seen:
            seen.add(t)
            entry_terms.append(t)

    return {"mesh_id": mesh_id, "preferred_label": preferred, "entry_terms": entry_terms}


/Users/lakshankarunathilake/miniconda3/envs/sapbert/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
mesh_english_aliases("D003920")

JSONDecodeError: Expecting value: line 3 column 1 (char 4)

In [6]:
import requests

def get_tgt(api_key: str) -> str:
    r = requests.post(
        "https://utslogin.nlm.nih.gov/cas/v1/api-key",
        data={"apikey": api_key},
        timeout=20,
    )
    r.raise_for_status()

    # Extract TGT URL from HTML response
    import re
    m = re.search(r'action="([^"]+)"', r.text)
    if not m:
        raise RuntimeError("Could not extract TGT")
    return m.group(1)

def get_service_ticket(tgt_url: str) -> str:
    r = requests.post(
        tgt_url,
        data={"service": "http://umlsks.nlm.nih.gov"},
        timeout=20,
    )
    r.raise_for_status()
    return r.text



def umls_english_aliases(cui: str, api_key: str) -> dict:
    tgt = get_tgt(api_key)
    st  = get_service_ticket(tgt)

    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}/atoms"
    params = {"ticket": st}

    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()

    names, seen = [], set()
    for atom in data["result"]:
        if atom.get("language") == "ENG":
            name = atom.get("name")
            if name and name not in seen:
                seen.add(name)
                names.append(name)

    return {
        "cui": cui,
        "english_aliases": names
    }


In [10]:
umls_english_aliases("D066126", "059c04d1-5174-471c-84f5-2eef7f4e175d")

HTTPError: 404 Client Error: Not Found for url: https://uts-ws.nlm.nih.gov/rest/content/current/CUI/D066126/atoms?ticket=ST-11991-krjfdf4q66mkwl0m3k-cas